# TCDL Institutional Reports

This Jupyter notebook contains the code to generate images,  text summaries, and Microsoft PowerPoint slide-decks for each institution's annual usage report. Currently, this report requires three sets of input files.

1. **Dataverse-users-{date}.xlsx - Dataverse-users-{date}.csv**: This file may either come in a CSV or an Excel format but should be converted to CSV for reading into the script (the string to identify the most recent version may need to be updated in the script). It contains a list of all users in TDR, including those not affiliated with any particular institution (the monthly institutional reports only contain affiliated users) and metadata about them (e.g., date of account creation, date of last API usage). It must be generated by an installation-level superuser (i.e. TDL).
2. **dataverse-reports-{date}**: This is a folder containing the most recent monthly reports for all TDR institutions (as well as UT Arlington, a former member). No additional processing is needed for these files; the script will load all of the files in and concatenate them together.
3. Outputs from the main assessment script that can be run by any regular TDR user. **{date}_all-institutions_all-datasets-combined-with-dataverses.csv** contains additional dataset-level metadata for published datasets. **{date}_all-institutions_all-dataverses.csv** contains additional collection-level metadata for published collections. **{date}_all-institutions_all-files-deduplicated-PUBLISHED.csv** contains additional file-level metadata for published files (though it normally excludes older versions / deleted versions of files that are not in the most recent published version).

## Workflow set-up

This codeblock contains the module import statements and loading of the requisite input dataframes. It is essentially the same as the loading steps for *tcdl-graphs.ipynb*, with some extraneous content removed.

In [1]:
#| echo: false
#| warning: false
#| include: false

import ast
import io
import json
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from datetime import date, datetime
from matplotlib.patches import Circle
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor
from utils import load_most_recent_file

# env file
with open('env.json', 'r') as file:
    env = json.load(file)

# Toggles
## Test environment (incomplete run, faster to complete)
test = env['TOGGLES']['test']
## Whether to exclue former/new members
current = env['TOGGLES']['current_members']

# Graphical parameters
img_width = env['GRAPHS']['default_width']
img_height = env['GRAPHS']['default_height']
dpi = env['GRAPHS']['default_dpi']
format = env['GRAPHS']['default_extension']
title_font = env['GRAPHS']['default_title']
axis_font = env['GRAPHS']['default_axis_titles']
tick_font = env['GRAPHS']['default_axis_labels']
legend_font = env['GRAPHS']['default_legend']

current_year = date.today().year

# Lists and dictionaries
school_colors = env['SCHOOL_MAPS']['SCHOOL_COLORS']
school_names = env['SCHOOL_MAPS']['SCHOOL_NAMES']
display_names = env['SCHOOL_MAPS']['DISPLAY_NAMES']
current_members = env['SCHOOL_MAPS']['MEMBERS']['current_members']
current_members_and_none = env['SCHOOL_MAPS']['MEMBERS']['current_members_plus']

# Get directories
script_dir = os.getcwd()
if test:
    outputs_dir = os.path.join(script_dir, 'test/outputs')
else:
    outputs_dir = os.path.join(script_dir, 'outputs')

# Define output directory for writing slide-decks
reports_dir = 'institutional-reports'
# Create directory if it doesn't exist
os.makedirs(reports_dir, exist_ok=True)

# Load most recent df of deposits
pattern = '_all-institutions_all-datasets-combined-with-dataverses.csv'
datasets = load_most_recent_file(outputs_dir, pattern)

pattern = '_all-institutions_all-dataverses.csv'
dataverses = load_most_recent_file(outputs_dir, pattern)

pattern = '_all-institutions_all-files-deduplicated-PUBLISHED.csv'
files = load_most_recent_file(outputs_dir, pattern)
files['category_mime_type'] = files['original_mime_type'].str.extract(r'(\w+)/')

pattern = 'Dataverse-users-2026'
users = load_most_recent_file(script_dir, pattern)

The most recent file '20260608_all-institutions_all-datasets-combined-with-dataverses.csv' has been loaded successfully.
The most recent file '20260608_all-institutions_all-dataverses.csv' has been loaded successfully.


c:\Users\bmg3525\OneDrive - The University of Texas at Austin\Documents\scripts\tdr-assessment-scripts\utils.py:362: DtypeWarning: Columns (3,25,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


The most recent file '20260608_all-institutions_all-files-deduplicated-PUBLISHED.csv' has been loaded successfully.
The most recent file 'Dataverse-users-20260428.xlsx - Dataverse-users-20260428.csv' has been loaded successfully.


In [2]:
# Standardize order of size bin labels (for datasets)
bin_labels = [
    '0-10 kB',
    '10 kB-1 MB',
    '1-100 MB',
    '100 MB-1 GB',
    '1-10 GB',
    '10-15 GB',
    '15-20 GB',
    '20-25 GB',
    '25-30 GB',
    '30-40 GB',
    '40-50 GB',
    '>50 GB'
]

In [3]:
# All datasets

# Users
## Create full name column
users['Full Name'] = users['First Name'] + ' ' + users['Last Name']
## Remove TDL / test accounts
users_real = users[~users['Email'].str.contains('mailinator.com|tdl.student', na=False) & ~users['Full Name'].str.contains('Lauland|Smutniak|Ryan Steans|Mumma|Admin|Nicholas Woodward', na=False, case=False)]

# Filter to remove certain institutions (former/recent TDR members)
if current:
    # Filter to keep only current members and 'No affiliation'
    files = files[files['institution'].isin(current_members_and_none)]
    datasets = datasets[datasets['institution_standardized'].isin(current_members_and_none)]
    dataverses = dataverses[dataverses['institution_standardized'].isin(current_members_and_none)]

# Files
## Drop any duplicate file types
files_dedup = files.drop_duplicates(subset=['doi', 'original_mime_type'])

# Standardizing institution affiliation
users_real['Affiliation_Standardized'] = users_real['Affiliation']
users_real['Affiliation_Standardized'] = users_real['Affiliation_Standardized'].replace(school_names)
users_real['Affiliation_Standardized'] = users_real['Affiliation_Standardized'].fillna('No affiliation')
## Replace affiliations not in the list with 'Other affiliation'
users_real['Affiliation_Standardized'] = users_real['Affiliation_Standardized'].apply(lambda x: x if x in current_members_and_none else 'Other affiliation')

## Remove sandbox datasets
datasets = datasets[~datasets['identifier'].str.startswith('FK2/')]
## Remove deaccessioned datasets
datasets_exist_df = datasets.dropna(subset=['createTime'])
## Remove draft datasets (but retain previously published datasets that are now in draft)
datasets_published_df = datasets_exist_df.dropna(subset=['publicationDate'])
## Get *only* unpublishe datasets
datasets_unpublished_df = datasets_exist_df[datasets_exist_df['publicationDate'].isna()]

## Remove parent institution-level collections
### Remove rows where dataverse_name is blank AND dataverseType is ORGANIZATIONS_INSTITUTIONS AND released is Yes
dataverses_actual = dataverses[~(
    (dataverses['dataverse_name'].isna() | (dataverses['dataverse_name'].str.strip() == '')) &
    (dataverses['dataverseType'] == 'ORGANIZATIONS_INSTITUTIONS') &
    (dataverses['released'] == 'Yes')
)]
## Remove unpublished dataverses
dataverses_published = dataverses_actual[dataverses_actual['released'] == 'Yes']

C:\Users\bmg3525\AppData\Local\Temp\ipykernel_21044\549101827.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  users_real['Affiliation_Standardized'] = users_real['Affiliation']
C:\Users\bmg3525\AppData\Local\Temp\ipykernel_21044\549101827.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  users_real['Affiliation_Standardized'] = users_real['Affiliation_Standardized'].replace(school_names)
C:\Users\bmg3525\AppData\Local\Temp\ipykernel_21044\549101827.py:23: SettingWithCopyWarning: 
A value is trying t

## Graphing functions

The following codeblock contains all of the code for recursively creating the relevant images. In most cases, the code is nearly identical to that used in the *tcdl-graphs.ipynb* file, with necessary modifications for making it into a function. In all of the functions, the variable 'institution_name' is not directly called and thus may appear to be inactive, but it needs to be retained.

In [4]:
def create_users_and_creators_trend(inst_data, institution_name):
    """
    Create cumulative trend chart for Users, Dataset creators, and Collection creators
    
    Parameters:
    - inst_data: filtered dataframe for the institution
    - institution_name: str, name of institution
    """
    
    fig, ax = plt.subplots(figsize=(img_width, img_height))
    
    # Define colors and line styles for each category
    styles = {
        'Users': {'color': '#3c68ee', 'linestyle': 'solid', 'linewidth': 3.5},
        'Dataset creators': {'color': '#05A00D', 'linestyle': 'dotted', 'linewidth': 3.5},
        'Collection creators': {'color': '#E4CF58', 'linestyle': 'dashed', 'linewidth': 3.5}
    }
    
    for category in inst_data['category'].unique():
        data = inst_data[inst_data['category'] == category]
        style = styles[category]
        
        ax.plot(data['year_month_created'], data['total'], 
                label=category,
                color=style['color'], 
                linestyle=style['linestyle'],
                linewidth=style['linewidth'])
    
    # Labels and formatting
    ax.set_xlabel('', fontsize=axis_font, fontweight='bold')
    ax.set_xlim(pd.Timestamp('2017-01-01'), pd.Timestamp('2026-05-31'))
    ax.set_ylabel('Cumulative Count', fontsize=axis_font, fontweight='bold')
    # ax.set_title('Users and Creators Over Time', fontsize=title_font, fontweight='bold')
    ax.tick_params(labelsize=tick_font)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    # Add legend
    ax.legend(fontsize=legend_font, loc='upper left')
    
    plt.tight_layout()

    return fig

def create_storage_allocation_trend(storage_summary, institution_name):
    """
    Create stacked area chart for cumulative storage allocation (Published vs Unpublished)
    
    Parameters:
    - storage_summary: dataframe with columns ['year_month', 'published', 'unpublished']
    - institution_name: str, name of institution
    """
    
    fig, ax = plt.subplots(figsize=(img_width, img_height))
    
    files_aligned = storage_summary['published']
    datasets_aligned = storage_summary['unpublished']
    
    ax.fill_between(storage_summary['year_month'], 0, files_aligned.values, alpha=0.6, color='#831ab8', label='Published')
    ax.fill_between(storage_summary['year_month'], files_aligned.values, files_aligned.values + datasets_aligned.values, alpha=0.6, color='#c299d7', label='Unpublished')
    
    # Plot the lines on top
    ax.plot(storage_summary['year_month'], files_aligned.values, color='#831ab8', linewidth=2.5)
    ax.plot(storage_summary['year_month'], files_aligned.values + datasets_aligned.values, color='#c299d7', linewidth=2.5)
    
    # Customize the plot
    ax.set_xlabel('', fontsize=axis_font, fontweight='bold')
    ax.set_xlim(pd.Timestamp('2017-01-01'), pd.Timestamp('2026-05-31'))
    ax.set_ylabel('Storage (TB)', fontsize=axis_font, fontweight='bold')
    # ax.set_title('Cumulative Storage Allocation over Time', fontsize=tick_font, fontweight='bold')
    ax.tick_params(labelsize=tick_font)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    ax.legend(fontsize=legend_font, loc='upper left')
    
    plt.tight_layout()

    return fig

def create_dataset_size_distribution(inst_datasets, institution_name, bin_labels):
    """
    Create dual-panel bar chart for dataset size distribution
    
    Parameters:
    - inst_datasets: filtered dataframe for the institution (datasets dataframe)
    - institution_name: str, name of institution
    - bin_labels: list, ordered list of dataset size bin labels
    """
    
    datasets_plot = inst_datasets.copy()
    
    # Counting datasets in each size bin
    datasets_by_size = datasets_plot.groupby(['dataset_size_bin']).size().reset_index(name='count')
    datasets_by_size = datasets_by_size.set_index('dataset_size_bin')
    datasets_by_size = datasets_by_size.reindex(bin_labels[::-1], fill_value=0)
    
    # Counting total storage size in each size bin
    datasets_size_sum = datasets_plot.groupby('dataset_size_bin')['dataset_size'].sum().reset_index(name='total_size')
    datasets_size_sum = datasets_size_sum.set_index('dataset_size_bin')
    datasets_size_sum = datasets_size_sum.reindex(bin_labels[::-1], fill_value=0)
    datasets_size_sum['total_size_tb'] = datasets_size_sum['total_size'] / (1024**4)
    
    # Create plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(img_width, img_height))
    
    # Only use colors for bins that have data
    num_bins = len(datasets_by_size)
    colors = plt.cm.nipy_spectral(np.linspace(0.9, 0, num_bins))
    
    y_pos = np.arange(len(datasets_by_size))
    
    # Left panel: Dataset count by size bin
    ax1.barh(y_pos, datasets_by_size['count'].values, 
            color=colors, edgecolor='black', linewidth=1.2)
    
    # Add value labels
    for i, count in enumerate(datasets_by_size['count'].values):
        if count > 0:
            ax1.text(count, i, f' {int(count)}', ha='left', va='center', fontsize=13)
    
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(datasets_by_size.index)
    ax1.set_xlabel('Count', fontsize=axis_font, fontweight='bold')
    ax1.set_ylabel('', fontsize=axis_font, fontweight='bold')
    ax1.set_title('Dataset Count', fontsize=title_font, fontweight='bold')
    ax1.tick_params(labelsize=tick_font)
    ax1.grid(axis='x', alpha=0.3, linestyle='--')
    ax1.set_axisbelow(True)
    
    # Add padding for labels
    x_max = datasets_by_size['count'].max()
    if x_max > 0:
        ax1.set_xlim(0, x_max * 1.2)
    
    # Right panel: Storage allocation by size bin
    y_pos = np.arange(len(datasets_size_sum))
    
    ax2.barh(y_pos, datasets_size_sum['total_size_tb'].values, 
            color=colors, edgecolor='black', linewidth=1.2)
    
    # Add value labels (convert TB to GB for display)
    for i, size_tb in enumerate(datasets_size_sum['total_size_tb'].values):
        if size_tb > 0:
            size_gb = size_tb * 1024
            ax2.text(size_tb, i, f' {size_gb:.1f} GB', ha='left', va='center', fontsize=13)
    
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(datasets_size_sum.index)
    ax2.set_xlabel('Storage (TB)', fontsize=axis_font, fontweight='bold')
    ax2.set_ylabel('', fontsize=axis_font, fontweight='bold')
    ax2.set_title('Storage Allocation', fontsize=title_font, fontweight='bold')
    ax2.tick_params(labelsize=tick_font)
    ax2.grid(axis='x', alpha=0.3, linestyle='--')
    ax2.set_axisbelow(True)
    
    # Add padding for labels
    x_max = datasets_size_sum['total_size_tb'].max()
    if x_max > 0:
        ax2.set_xlim(0, x_max * 1.3)
    
    # Add space between plots
    plt.subplots_adjust(wspace=0.3)
    plt.tight_layout()
    
    return fig

def create_collections_trend(inst_dataverses, institution_name):
    """
    Create cumulative trend chart for Collections (published vs created)
    
    Parameters:
    - inst_dataverses: filtered dataframe for the institution (dataverses dataframe)
    - institution_name: str, name of institution
    """
    
    # Count records per month - published
    monthly_counts_published = inst_dataverses.groupby('YearMonth_published').size()
    # Count records per month - creation
    monthly_counts_creation = inst_dataverses.groupby('year_month_created').size()
    
    # Convert to cumulative sum
    cumulative_counts_published = monthly_counts_published.cumsum()
    cumulative_counts_creation = monthly_counts_creation.cumsum()
    
    # Convert period index to timestamp for plotting
    cumulative_counts_published.index = pd.to_datetime(cumulative_counts_published.index)
    cumulative_counts_creation.index = pd.to_datetime(cumulative_counts_creation.index)
    
    # Create plot
    fig, ax = plt.subplots(figsize=(img_width, img_height))
    
    # Plot both lines
    ax.plot(cumulative_counts_published.index, cumulative_counts_published.values, 
            color='#AB6209', linewidth=3.5, label='Published')
    
    ax.plot(cumulative_counts_creation.index, cumulative_counts_creation.values, 
            color='#DBAC00', linewidth=3.5, linestyle='dashed', label='Created')
    
    # Customize the plot
    ax.set_xlabel('', fontsize=axis_font, fontweight='bold')
    ax.set_xlim(pd.Timestamp('2017-01-01'), pd.Timestamp('2026-05-31'))
    ax.set_ylabel('Count', fontsize=axis_font, fontweight='bold')
    # ax.set_title('Cumulative Collections Over Time', fontsize=title_font, fontweight='bold')
    ax.tick_params(labelsize=tick_font)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=legend_font, loc='upper left')
    
    # Adjust layout
    plt.tight_layout()
    
    return fig

def create_datasets_trend(inst_datasets, institution_name):
    """
    Create cumulative trend chart for Datasets (created vs published)
    
    Parameters:
    - inst_datasets: filtered dataframe for the institution (datasets dataframe)
    - institution_name: str, name of institution
    """
    
    # Datasets created - count by creation date
    monthly_counts_created = inst_datasets.groupby('year_month_created').size()
    cumulative_counts_created = monthly_counts_created.cumsum()
    cumulative_counts_created.index = pd.to_datetime(cumulative_counts_created.index)
    
    dataset_create_counts = cumulative_counts_created.reset_index()
    dataset_create_counts.columns = ['year_month', 'total']
    dataset_create_counts['category'] = 'Created'
    
    # Datasets published - count by publication date
    monthly_counts_published = inst_datasets.groupby('year_month_published').size()
    cumulative_counts_published = monthly_counts_published.cumsum()
    cumulative_counts_published.index = pd.to_datetime(cumulative_counts_published.index)
    
    dataset_publish_counts = cumulative_counts_published.reset_index()
    dataset_publish_counts.columns = ['year_month', 'total']
    dataset_publish_counts['category'] = 'Published'
    
    # Concatenate
    dataset_concat = pd.concat([dataset_publish_counts, dataset_create_counts], ignore_index=True)
    
    # Create plot
    fig, ax = plt.subplots(figsize=(img_width, img_height))
    
    # Define colors and line styles for each category
    styles = {
        'Created': {'color': '#a9ee8e', 'linestyle': 'dashed', 'linewidth': 3.5},
        'Published': {'color': '#47b661', 'linestyle': 'solid', 'linewidth': 3.5}
    }
    
    for category in dataset_concat['category'].unique():
        data = dataset_concat[dataset_concat['category'] == category]
        style = styles[category]
        
        ax.plot(data['year_month'], data['total'], 
                label=category,
                color=style['color'], 
                linestyle=style['linestyle'],
                linewidth=style['linewidth'])
    
    # Labels, etc.
    ax.set_xlabel('', fontsize=axis_font, fontweight='bold')
    ax.set_xlim(pd.Timestamp('2017-01-01'), pd.Timestamp('2026-05-31'))
    ax.set_ylabel('Count', fontsize=axis_font, fontweight='bold')
    # ax.set_title('Cumulative Datasets Over Time', fontsize=title_font, fontweight='bold')
    ax.tick_params(labelsize=tick_font)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=legend_font, loc='upper left')
    
    plt.tight_layout()
    
    return fig

def create_datasets_by_subject_bar(inst_datasets, institution_name):
    """
    Create horizontal bar chart for datasets by subject
    
    Parameters:
    - inst_datasets: filtered dataframe for the institution (datasets dataframe)
    - institution_name: str, name of institution
    """
    
    datasets_plot = inst_datasets.copy()
    
    # Parse subjects from string to list
    datasets_plot['subjects'] = datasets_plot['subjects'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    
    # Explode to split multi-subject datasets
    df_exploded = datasets_plot.explode('subjects')
    
    # Clean whitespace and drop NaN
    df_exploded['subjects'] = df_exploded['subjects'].str.strip()
    df_exploded = df_exploded.dropna(subset=['subjects'])
    
    # Count datasets by subject
    datasets_by_subject = df_exploded['subjects'].value_counts().sort_values(ascending=True)
    
    # Create plot
    fig, ax = plt.subplots(figsize=(img_width, img_height))
    
    # Generate colors
    num_colors = len(datasets_by_subject)
    if num_colors > 0:
        colors = plt.cm.tab20b(np.linspace(0.9, 0, num_colors))
        
        # Create horizontal bar chart using matplotlib directly
        y_pos = np.arange(len(datasets_by_subject))
        ax.barh(y_pos, datasets_by_subject.values, color=colors, edgecolor='black', linewidth=1.2)
        
        # Add value labels
        for i, value in enumerate(datasets_by_subject.values):
            ax.text(value, i, f' {int(value)}', va='center', fontsize=13)
        
        # Set labels
        ax.set_yticks(y_pos)
        ax.set_yticklabels(datasets_by_subject.index)
    
    ax.set_xlabel('Count', fontsize=axis_font, fontweight='bold')
    ax.set_ylabel('', fontsize=axis_font, fontweight='bold')
    # ax.set_title('Published Datasets by Subject', fontsize=title_font, fontweight='bold')
    ax.tick_params(labelsize=tick_font)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    
    return fig

def create_file_formats_pie(inst_files, institution_name, published_dataset_count):
    """
    Create donut chart for most common file formats by unique dataset count
    
    Parameters:
    - inst_files: filtered dataframe for the institution (files dataframe)
    - institution_name: str, name of institution
    - published_dataset_count: int, count of published datasets for center text
    """
    
    files_plot = inst_files.copy()
    
    files_plots_dedup = files_plot.drop_duplicates(subset=['doi', 'friendly_format_manual'], keep='first')
    
    files_by_dataset = files_plots_dedup['friendly_format_manual'].groupby(files_plot['friendly_format_manual']).count()
    files_by_dataset = files_by_dataset.sort_values(ascending=False)
    
    cut = 15
    top = files_by_dataset.head(cut)
    other_count = files_by_dataset.iloc[cut:].sum()
    
    # Create new series with 'Other' category
    if other_count > 0:
        files_by_dataset = pd.concat([top, pd.Series({'Other': other_count})])
    else:
        files_by_dataset = top
    
    fig, ax = plt.subplots(figsize=(img_width, img_height))
    
    colors = plt.cm.tab20(range(len(files_by_dataset)))
    
    wedges, texts, autotexts = ax.pie(
        files_by_dataset.values,
        autopct='%1.1f%%',
        colors=colors,
        startangle=90,
        textprops={'fontsize': 10, 'weight': 'bold'},
        wedgeprops={'edgecolor': 'black', 'linewidth': 1}
    )
    
    border = Circle((0, 0), 1, fill=False, edgecolor='black', linewidth=1.5)
    ax.add_patch(border)
    
    # Make percentage text white and bold
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
        autotext.set_fontsize(9)
    
    # Remove the label text around the donut
    for text in texts:
        text.set_visible(False)
    
    # Draw circle in the center to create donut effect
    centre_circle = plt.Circle((0, 0), 0.70, fc='white', color='black', linewidth=1.5)
    ax.add_artist(centre_circle)
    
    # Add count to middle
    ax.text(0, 0, f'{published_dataset_count:,} published datasets', ha='center', va='center', 
            fontsize=15, fontweight='bold', color='#5C5252')
    
    # Create legend with labels and percentages
    legend_labels = [f'{label} ({value/files_by_dataset.sum()*100:.1f}%)' 
                     for label, value in files_by_dataset.items()]
    ax.legend(legend_labels, loc='center left', bbox_to_anchor=(1, 0, 0.5, 1), fontsize=12)
    
    # ax.set_title('Most Common File Formats (by Unique Dataset Count)', fontsize=title_font, fontweight='bold')
    plt.tight_layout()
    
    return fig

def create_file_formats_by_count_pie(inst_files, institution_name):
    """
    Create donut chart for most common file formats by total file count
    
    Parameters:
    - inst_files: filtered dataframe for the institution (files dataframe)
    - institution_name: str, name of institution
    """
    
    files_plot = inst_files.copy()
    
    files_by_type = files_plot['friendly_format_manual'].groupby(files_plot['friendly_format_manual']).count()
    files_by_type = files_by_type.sort_values(ascending=False)
    
    cut = 15
    top = files_by_type.head(cut)
    other_count = files_by_type.iloc[cut:].sum()
    
    # Create new series with 'Other' category
    if other_count > 0:
        files_by_type = pd.concat([top, pd.Series({'Other': other_count})])
    else:
        files_by_type = top
    
    fig, ax = plt.subplots(figsize=(img_width, img_height))
    
    colors = plt.cm.tab20(range(len(files_by_type)))
    
    wedges, texts, autotexts = ax.pie(
        files_by_type.values,
        autopct='%1.1f%%',
        colors=colors,
        startangle=90,
        textprops={'fontsize': 10, 'weight': 'bold'},
        wedgeprops={'edgecolor': 'black', 'linewidth': 1}
    )
    
    border = Circle((0, 0), 1, fill=False, edgecolor='black', linewidth=1.5)
    ax.add_patch(border)
    
    # Make percentage text white and bold
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
        autotext.set_fontsize(9)
    
    # Remove the label text around the donut
    for text in texts:
        text.set_visible(False)
    
    # Draw circle in the center to create donut effect
    centre_circle = plt.Circle((0, 0), 0.70, fc='white', color='black', linewidth=1.5)
    ax.add_artist(centre_circle)
    
    # Add count to middle
    total_count = len(files_plot)
    ax.text(0, 0, f'{total_count:,} published files', ha='center', va='center', 
            fontsize=15, fontweight='bold', color='#5C5252')
    
    # Create legend with labels and percentages
    legend_labels = [f'{label} ({value/files_by_type.sum()*100:.1f}%)' 
                     for label, value in files_by_type.items()]
    ax.legend(legend_labels, loc='center left', bbox_to_anchor=(1, 0, 0.5, 1), fontsize=12)
    
    # ax.set_title('Most Common File Formats (by Total File Count)', fontsize=title_font, fontweight='bold')
    plt.tight_layout()
    
    return fig

def create_dataset_downloads_bin(inst_datasets, institution_name):
    """
    Create horizontal bar chart for dataset downloads by bin
    
    Parameters:
    - inst_datasets: filtered dataframe for the institution (datasets dataframe)
    - institution_name: str, name of institution
    """
    
    datasets_plot = inst_datasets.copy()
    
    # Create bins for downloads
    def bin_downloads(count):
        if pd.isna(count):
            return 'Unpublished'
        elif count == 0:
            return '0 downloads'
        elif count <= 10:
            return '1-10 downloads'
        elif count <= 20:
            return '11-20 downloads'
        elif count <= 30:
            return '21-30 downloads'
        elif count <= 40:
            return '31-40 downloads'
        elif count <= 50:
            return '41-50 downloads'
        elif count <= 100:
            return '51-100 downloads'
        elif count <= 150:
            return '101-150 downloads'
        elif count <= 200:
            return '151-200 downloads'
        else:
            return '>200 downloads'
    
    datasets_plot['downloads_bin'] = datasets_plot['downloads_dc'].apply(bin_downloads)
    downloads_count = datasets_plot['downloads_bin'].value_counts()
    
    category_order = [
        '0 downloads',
        '1-10 downloads',
        '11-20 downloads',
        '21-30 downloads',
        '31-40 downloads',
        '41-50 downloads',
        '51-100 downloads',
        '101-150 downloads',
        '151-200 downloads',
        '>200 downloads'
    ]
    
    # Reorder the series
    downloads_count = downloads_count.reindex(category_order, fill_value=0)
    n_categories = len(category_order)
    
    # Create plot
    fig, ax = plt.subplots(figsize=(7, img_height))
    
    # Create horizontal bar plot
    colors = cm.Spectral(np.linspace(0.1, 0.9, n_categories))
    bars = ax.barh(range(len(downloads_count)), downloads_count.values, 
                   color=colors, edgecolor='black', linewidth=1.2)
    
    # Customize the plot
    ax.set_ylabel('', fontsize=axis_font, fontweight='bold')
    ax.set_xlabel('Count', fontsize=axis_font, fontweight='bold')
    ax.set_title('Dataset Downloads by Bin', fontsize=title_font, fontweight='bold')
    ax.tick_params(labelsize=tick_font)
    
    # Set y-axis labels
    ax.set_yticks(range(len(downloads_count)))
    ax.set_yticklabels(downloads_count.index)
    
    # Add value labels at the end of bars
    for idx, (bar, value) in enumerate(zip(bars, downloads_count.values)):
        if value > 0:  # Only show label if value > 0
            width = bar.get_width()
            pct = (value / downloads_count.sum()) * 100
            ax.text(width, bar.get_y() + bar.get_height()/2.,
                    f' {int(value)} ({pct:.1f}%)',
                    ha='left', va='center', fontsize=13)
    
    # Add grid for readability
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    # Set x-axis limit with padding
    x_max = downloads_count.max()
    if x_max > 0:
        ax.set_xlim(0, x_max * 1.45)
    
    plt.tight_layout()
    
    return fig

def create_dataset_views_bin(inst_datasets, institution_name):
    """
    Create horizontal bar chart for dataset views by bin
    
    Parameters:
    - inst_datasets: filtered dataframe for the institution (datasets dataframe)
    - institution_name: str, name of institution
    """
    
    datasets_plot = inst_datasets.copy()
    
    # Create bins for views
    def bin_views(count):
        if pd.isna(count):
            return 'Unpublished'
        elif count == 0:
            return '0 views'
        elif count <= 100:
            return '1-100 views'
        elif count <= 250:
            return '101-250 views'
        elif count <= 500:
            return '251-500 views'
        elif count <= 750:
            return '501-750 views'
        elif count <= 1000:
            return '751-1000 views'
        elif count <= 2000:
            return '1001-2000 views'
        elif count <= 3000:
            return '2001-3000 views'
        elif count <= 4000:
            return '3001-4000 views'
        else:
            return '>4000 views'
    
    datasets_plot['views_bin'] = datasets_plot['views_dc'].apply(bin_views)
    views_count = datasets_plot['views_bin'].value_counts()
    
    category_order = [
        '0 views',
        '1-100 views',
        '101-250 views',
        '251-500 views',
        '501-750 views',
        '751-1000 views',
        '1001-2000 views',
        '2001-3000 views',
        '3001-4000 views',
        '>4000 views'
    ]
    
    # Reorder the series
    views_count = views_count.reindex(category_order, fill_value=0)
    n_categories = len(category_order)
    
    # Create plot
    fig, ax = plt.subplots(figsize=(7, img_height))
    
    colors = cm.PiYG(np.linspace(0.1, 0.9, n_categories))
    bars = ax.barh(range(len(views_count)), views_count.values, 
                   color=colors, edgecolor='black', linewidth=1.2)
    
    # Labels, etc.
    ax.set_ylabel('', fontsize=12, fontweight='bold')
    ax.set_xlabel('Count', fontsize=axis_font, fontweight='bold')
    ax.set_title('Dataset Views by Bin', fontsize=title_font, fontweight='bold')
    ax.tick_params(labelsize=tick_font)
    ax.set_yticks(range(len(views_count)))
    ax.set_yticklabels(views_count.index)
    
    # Add value labels at the end of bars
    for idx, (bar, value) in enumerate(zip(bars, views_count.values)):
        if value > 0:  # Only show label if value > 0
            width = bar.get_width()
            pct = (value / views_count.sum()) * 100
            ax.text(width, bar.get_y() + bar.get_height()/2.,
                    f' {int(value)} ({pct:.1f}%)',
                    ha='left', va='center', fontsize=13)
    
    # Add grid for readability
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    # Set x-axis limit with padding
    x_max = views_count.max()
    if x_max > 0:
        ax.set_xlim(0, x_max * 1.45)
    
    plt.tight_layout()
    
    return fig

def add_dataset_citation_table(slide, datasets_df, top_position, left_position, width, height, top_n=10):
    """
    Add a table to a slide showing top datasets by citations with hyperlinked titles.
    """
    
    # Sort by citations in descending order, handling NaN values, and limit to top_n
    df_sorted = datasets_df.dropna(subset=['citations_dc'])
    df_sorted = df_sorted[df_sorted['citations_dc'] > 0].sort_values('citations_dc', ascending=False).head(top_n)
    
    # Create table with 2 columns and (rows + 1 for header)
    rows = len(df_sorted) + 1
    cols = 2
    
    table_shape = slide.shapes.add_table(rows, cols, 
                                         Inches(left_position), 
                                         Inches(top_position), 
                                         Inches(width), 
                                         Inches(height)).table
    
    # Set column widths
    table_shape.columns[0].width = Inches(width * 0.85)  # Dataset title gets 85%
    table_shape.columns[1].width = Inches(width * 0.15)  # Citations gets 15%
    
    # Add header row
    header_cells = [table_shape.cell(0, 0), table_shape.cell(0, 1)]
    headers = ['Dataset Title', 'Citations']
    
    for cell, header_text in zip(header_cells, headers):
        cell.text = header_text
        # Format header
        paragraph = cell.text_frame.paragraphs[0]
        paragraph.font.bold = True
        paragraph.font.size = Pt(24)
        paragraph.font.color.rgb = RGBColor(255, 255, 255)
        # Header background color (dark blue)
        cell.fill.solid()
        cell.fill.fore_color.rgb = RGBColor(31, 78, 121)

    # Add data rows
    for row_idx, (_, row_data) in enumerate(df_sorted.iterrows(), start=1):
        # Dataset title with hyperlink
        title_cell = table_shape.cell(row_idx, 0)
        paragraph = title_cell.text_frame.paragraphs[0]
        
        # Create hyperlink
        if pd.notna(row_data['persistentUrl']):
            run = paragraph.add_run()
            run.text = str(row_data['dataset_title'])
            run.hyperlink.address = row_data['persistentUrl']  # Use .address instead
            
            # Style the hyperlink
            run.font.underline = True
            run.font.size = Pt(20)
            run.font.color.rgb = RGBColor(0, 0, 255)
        else:
            # No hyperlink, just add text
            run = paragraph.add_run()
            run.text = str(row_data['dataset_title'])
            run.font.size = Pt(20)
        
        # Citations count (handle NaN)
        citations_cell = table_shape.cell(row_idx, 1)
        citations_value = int(row_data['citations_dc']) if pd.notna(row_data['citations_dc']) else 0
        citations_cell.text = str(citations_value)
        citations_cell.text_frame.paragraphs[0].font.size = Pt(20)

def get_figure_bottom_position(picture_shape, picture_top_inches):
    """
    Calculate the bottom vertical position of a picture on the slide
    
    Parameters:
    - picture_shape: the shape object returned by add_picture()
    - picture_top_inches: the top position in inches where the picture was placed
    
    Returns:
    - bottom_position: float, the bottom edge position in inches
    """
    # Convert picture height from EMU (English Metric Units) to inches
    picture_height_inches = picture_shape.height / 914400
    bottom_position = picture_top_inches + picture_height_inches
    return bottom_position

def get_table_bottom_position(table_top, table_height_inches):
    """
    Calculate the bottom Y position of a table.
    
    Parameters:
    - table_shape: The table shape object
    - table_top: The top Y position in inches
    - table_height_inches: The height of the table in inches
    
    Returns:
    - Bottom Y position in inches
    """
    return table_top + table_height_inches

## Slide-deck creation function

The following codeblock contains all of the code for creating a slide-deck. It is fairly long since it manually defines each slide (though most of the images have nearly identical formatting).

In [ ]:
def create_presentation(institution_name, display_names, figures, dataset_citation_table, output_path, institution_color, total_users=0, total_dataverses=0, total_datasets=0, published_datasets=0, unpublished_datasets=0, published_files=0, published_storage=0, all_storage=0):
    """
    Create a PowerPoint presentation for an institution
    
    Parameters:
    - institution_name: str, name of institution in dataframes
    - display_names: str, name of institution to be displayed on title slide
    - figures: dict, with keys 'users' and 'storage' containing matplotlib figures,
    - dataset_citation_table: single table for most cited datasets
    - output_path: str, path to save the .pptx file
    - institution_color: str, hex color code for the institution
    - total_users, total_dataverses, etc.: filler values set to 0 for all summary statistics counters
    """

    # Convert hex to RGB for PowerPoint
    rgb_color = tuple(int(institution_color.lstrip('#')[i:i+2], 16) for i in (0, 2, 4))
    
    # Get display (official) names
    display_name = display_names.get(institution_name, institution_name)

    # Create presentation
    prs = Presentation()
    prs.slide_width = Inches(16)    # was 10 inches
    prs.slide_height = Inches(9)   # was 7.5 inches
    
    ##################################
    # ===== SLIDE 1: Title Slide =====
    ##################################

    slide1 = prs.slides.add_slide(prs.slide_layouts[6])
    background = slide1.background
    fill = background.fill
    fill.solid()
    fill.fore_color.rgb = RGBColor(*rgb_color)
    
    # Add title
    title_box = slide1.shapes.add_textbox(Inches(0.5), Inches(2.5), Inches(img_width), Inches(2))
    title_frame = title_box.text_frame
    title_frame.word_wrap = True
    p = title_frame.paragraphs[0]
    p.text = display_name
    p.font.size = Pt(60)
    p.font.bold = True
    p.font.color.rgb = RGBColor(255, 255, 255)
    p.alignment = PP_ALIGN.CENTER
    
    # Add subtitle
    subtitle_box = slide1.shapes.add_textbox(Inches(0.5), Inches(4.5), Inches(img_width), Inches(1))
    subtitle_frame = subtitle_box.text_frame
    p = subtitle_frame.paragraphs[0]
    p.text = f"TDR Annual Usage Report - {datetime.now().strftime('%B %Y')}"
    p.font.size = Pt(24)
    p.font.color.rgb = RGBColor(255, 255, 255)
    p.alignment = PP_ALIGN.CENTER
    
    #########################################
    # ===== SLIDE 2: Summary Statistics =====
    #########################################

    slide_summary = prs.slides.add_slide(prs.slide_layouts[6])

    # Add title
    title_box = slide_summary.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(img_width), Inches(0.6))
    title_frame = title_box.text_frame
    p = title_frame.paragraphs[0]
    p.text = 'Summary Statistics'
    p.font.size = Pt(32)
    p.font.bold = True
    p.font.color.rgb = RGBColor(*rgb_color)

    # Add bullet points
    text_box = slide_summary.shapes.add_textbox(Inches(1), Inches(1.2), Inches(8), Inches(5.5))
    text_frame = text_box.text_frame
    text_frame.word_wrap = True

    bullet_points = [
        f'Total user count: {total_users:,}',
        f'Total collection count: {total_dataverses:,}',
        f'Total dataset count: {total_datasets:,}',
        f'Published dataset count: {published_datasets:,}',
        f'Unpublished dataset count: {unpublished_datasets:,}',
        f'Published file count: {published_files:,}',
        f'Total storage: {all_storage} TB',
        f'Total published storage: {published_storage:,} TB'
    ]

    for i, bullet in enumerate(bullet_points):
        if i == 0:
            p = text_frame.paragraphs[0]
        else:
            p = text_frame.add_paragraph()
        p.level = 0
        p.space_before = Pt(12)
        
        # Split the text at the colon
        label, value = bullet.split(': ', 1)
        
        # Add the bold label
        run = p.add_run()
        run.text = label + ': '
        run.font.bold = True
        run.font.size = Pt(24)
        
        # Add the value (not bold)
        run = p.add_run()
        run.text = value
        run.font.size = Pt(24)

    ###############################################
    # ===== SLIDE 3: Users and Creators Trend =====
    ###############################################

    slide = prs.slides.add_slide(prs.slide_layouts[6])

    # Add title
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(img_width), Inches(0.6))
    title_frame = title_box.text_frame
    p = title_frame.paragraphs[0]
    p.text = 'Users and Creators Over Time'
    p.font.size = Pt(32)
    p.font.bold = True
    p.font.color.rgb = RGBColor(*rgb_color)

    # Add figure with alt text
    fig_users = figures['users']
    img_stream = io.BytesIO()
    fig_users.savefig(img_stream, format='png',dpi = 100, bbox_inches='tight')
    img_stream.seek(0)
    picture_top = 1.1
    picture = slide.shapes.add_picture(img_stream, Inches(0.5), Inches(picture_top), width=Inches(img_width))
    picture.name = f'{institution_name} Users and Creators Trend'
    picture._element._nvXxPr.cNvPr.attrib['descr'] = f'Cumulative counts of users, dataset creators, and collection creators for {institution_name} over time.'

    # Add caption
    caption_y = get_figure_bottom_position(picture, picture_top) + 0.1
    caption_box = slide.shapes.add_textbox(Inches(0.5), Inches(caption_y), Inches(img_width), Inches(0.8))    
    caption_frame = caption_box.text_frame
    caption_frame.word_wrap = True
    p = caption_frame.paragraphs[0]
    p.text = 'Figure 1: Cumulative counts of users, dataset creators, and collection creators over time.'
    p.font.size = Pt(18)
    p.font.italic = True
    p.font.color.rgb = RGBColor(100, 100, 100)

    #########################################
    # ===== SLIDE 4: Storage Allocation =====
    #########################################

    slide = prs.slides.add_slide(prs.slide_layouts[6])

    # Add title
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(img_width), Inches(0.6))
    title_frame = title_box.text_frame
    p = title_frame.paragraphs[0]
    p.text = 'Cumulative Storage Allocation'
    p.font.size = Pt(32)
    p.font.bold = True
    p.font.color.rgb = RGBColor(*rgb_color)

    # Add figure with alt text
    fig_storage = figures['storage']
    img_stream = io.BytesIO()
    fig_storage.savefig(img_stream, format='png',dpi = 100, bbox_inches='tight')
    img_stream.seek(0)
    picture_top = 1.1
    picture = slide.shapes.add_picture(img_stream, Inches(0.5), Inches(picture_top), width=Inches(img_width))
    picture.name = f'{institution_name} Storage Allocation Trend'
    picture._element._nvXxPr.cNvPr.attrib['descr'] = f'Cumulative published and unpublished storage allocation for {institution_name} over time. The total height (y-axis position) of the combined line fills represents the total.'

    # Add caption
    caption_y = get_figure_bottom_position(picture, picture_top) + 0.1
    caption_box = slide.shapes.add_textbox(Inches(0.5), Inches(caption_y), Inches(img_width), Inches(0.8))
    caption_frame = caption_box.text_frame
    caption_frame.word_wrap = True
    p = caption_frame.paragraphs[0]
    p.text = 'Figure 2: Cumulative storage allocation for published and unpublished datasets over time.'
    p.font.size = Pt(18)
    p.font.italic = True
    p.font.color.rgb = RGBColor(100, 100, 100)

    ###########################################
    # ===== SLIDE 5: Datasets by Size Bin =====
    ###########################################
    
    slide = prs.slides.add_slide(prs.slide_layouts[6])

    # Add title
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(img_width), Inches(0.6))
    title_frame = title_box.text_frame
    p = title_frame.paragraphs[0]
    p.text = 'Datasets by Size Bin'
    p.font.size = Pt(32)
    p.font.bold = True
    p.font.color.rgb = RGBColor(*rgb_color)

    # Add figure with alt text
    fig_storage = figures['dataset_size']
    img_stream = io.BytesIO()
    fig_storage.savefig(img_stream, format='png',dpi = 100, bbox_inches='tight')
    img_stream.seek(0)
    picture_top = 1.1
    picture = slide.shapes.add_picture(img_stream, Inches(0.5), Inches(picture_top), width=Inches(img_width))
    picture.name = f'{institution_name} Datasets by Size Bin'
    picture._element._nvXxPr.cNvPr.attrib['descr'] = f'Count of datasets by size bin and total storage of datasets within each size bin for {institution_name}.'

    # Add caption
    caption_y = get_figure_bottom_position(picture, picture_top) + 0.1
    caption_box = slide.shapes.add_textbox(Inches(0.5), Inches(caption_y), Inches(img_width), Inches(0.8))
    caption_frame = caption_box.text_frame
    caption_frame.word_wrap = True
    p = caption_frame.paragraphs[0]
    p.text = 'Figure 3: Datasets by size bin (count and total storage size).'
    p.font.size = Pt(18)
    p.font.italic = True
    p.font.color.rgb = RGBColor(100, 100, 100)

    ##################################
    # ===== SLIDE 6: Collections =====
    ##################################

    slide = prs.slides.add_slide(prs.slide_layouts[6])

    # Add title
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(img_width), Inches(0.6))
    title_frame = title_box.text_frame
    p = title_frame.paragraphs[0]
    p.text = 'Collection Creation & Publication Over Time'
    p.font.size = Pt(32)
    p.font.bold = True
    p.font.color.rgb = RGBColor(*rgb_color)

    # Add figure with alt text and name
    fig_storage = figures['collections']
    img_stream = io.BytesIO()
    fig_storage.savefig(img_stream, format='png',dpi = 100, bbox_inches='tight')
    img_stream.seek(0)
    picture_top = 1.1
    picture = slide.shapes.add_picture(img_stream, Inches(0.5), Inches(picture_top), width=Inches(img_width))
    picture.name = f'{institution_name} Collection Trend'
    picture._element._nvXxPr.cNvPr.attrib['descr'] = f'Line chart showing cumulative collection creation and publication over time for {institution_name}. The chart displays two lines tracking collections created versus collections published from 2017 to 2026.'

    # Add caption
    caption_y = get_figure_bottom_position(picture, picture_top) + 0.1
    caption_box = slide.shapes.add_textbox(Inches(0.5), Inches(caption_y), Inches(img_width), Inches(0.8))
    caption_frame = caption_box.text_frame
    caption_frame.word_wrap = True
    p = caption_frame.paragraphs[0]
    p.text = 'Figure 4: Cumulative collection creation and publication over time.'
    p.font.size = Pt(18)
    p.font.italic = True
    p.font.color.rgb = RGBColor(100, 100, 100)

    ###############################
    # ===== SLIDE 7: Datasets =====
    ###############################
    
    slide = prs.slides.add_slide(prs.slide_layouts[6])

    # Add title
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(img_width), Inches(0.6))
    title_frame = title_box.text_frame
    p = title_frame.paragraphs[0]
    p.text = 'Dataset Creation & Publication Over Time'
    p.font.size = Pt(32)
    p.font.bold = True
    p.font.color.rgb = RGBColor(*rgb_color)

    # Add figure with alt text
    fig_storage = figures['datasets']
    img_stream = io.BytesIO()
    fig_storage.savefig(img_stream, format='png',dpi = 100, bbox_inches='tight')
    img_stream.seek(0)
    picture_top = 1.1
    picture = slide.shapes.add_picture(img_stream, Inches(0.5), Inches(picture_top), width=Inches(img_width))
    picture.name = f'{institution_name} Dataset Trend'
    picture._element._nvXxPr.cNvPr.attrib['descr'] = f'Line chart showing cumulative dataset creation and publication over time for {institution_name}. The chart displays two lines tracking datasets created versus datasets published from 2017 to 2026.'

    # Add caption
    caption_y = get_figure_bottom_position(picture, picture_top) + 0.1
    caption_box = slide.shapes.add_textbox(Inches(0.5), Inches(caption_y), Inches(img_width), Inches(0.8))
    caption_frame = caption_box.text_frame
    caption_frame.word_wrap = True
    p = caption_frame.paragraphs[0]
    p.text = 'Figure 5: Cumulative dataset creation and publication over time.'
    p.font.size = Pt(18)
    p.font.italic = True
    p.font.color.rgb = RGBColor(100, 100, 100)
    
    ##########################################
    # ===== SLIDE 8: Datasets by Subject =====
    ###########################################

    slide = prs.slides.add_slide(prs.slide_layouts[6])

    # Add title
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(img_width), Inches(0.6))
    title_frame = title_box.text_frame
    p = title_frame.paragraphs[0]
    p.text = 'Published Datasets by Subject'
    p.font.size = Pt(32)
    p.font.bold = True
    p.font.color.rgb = RGBColor(*rgb_color)

    # Add figure with alt text
    fig_storage = figures['datasets_subject']
    img_stream = io.BytesIO()
    fig_storage.savefig(img_stream, format='png',dpi = 100, bbox_inches='tight')  # Changed from 150
    img_stream.seek(0)
    picture_top = 1.1
    picture = slide.shapes.add_picture(img_stream, Inches(0.5), Inches(picture_top), width=Inches(img_width))
    picture.name = f'{institution_name} Datasets by Subject'
    picture._element._nvXxPr.cNvPr.attrib['descr'] = f'Horizontal bar graph comparing frequency of subjects among published datasets for {institution_name}.'
    
    # Add caption
    caption_y = get_figure_bottom_position(picture, picture_top) + 0.1
    caption_box = slide.shapes.add_textbox(Inches(0.5), Inches(caption_y), Inches(13), Inches(0.8))
    caption_frame = caption_box.text_frame
    caption_frame.word_wrap = True
    p = caption_frame.paragraphs[0]
    p.text = 'Figure 6: Published datasets by subject. Multi-subject datasets are counted for each one.'
    p.font.size = Pt(18)
    p.font.italic = True
    p.font.color.rgb = RGBColor(100, 100, 100)

    ######################################################
    # ===== SLIDE 9: File Format by Total File Count =====
    ######################################################

    slide = prs.slides.add_slide(prs.slide_layouts[6])

    # Add title
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(img_width), Inches(0.6))
    title_frame = title_box.text_frame
    p = title_frame.paragraphs[0]
    p.text = 'Most Common File Formats (by Total File Count)'
    p.font.size = Pt(32)
    p.font.bold = True
    p.font.color.rgb = RGBColor(*rgb_color)

    # Add figure with alt text
    fig_storage = figures['formats_by_count']
    img_stream = io.BytesIO()
    # fig_storage.savefig(img_stream, format='png',dpi = 100, bbox_inches='tight')
    fig_storage.savefig(img_stream, format='png',dpi = 100)
    img_stream.seek(0)
    picture_top = 1.1
    picture = slide.shapes.add_picture(img_stream, Inches(0.5), Inches(picture_top), width=Inches(img_width))
    picture.name = f'{institution_name} File Formats by Total File Count'
    picture._element._nvXxPr.cNvPr.attrib['descr'] = f'Donut graph comparing the 15 most common file formats by total file count for {institution_name}.'

    # Add caption
    caption_y = get_figure_bottom_position(picture, picture_top) + 0.1
    caption_box = slide.shapes.add_textbox(Inches(0.5), Inches(caption_y), Inches(img_width), Inches(0.8))
    caption_frame = caption_box.text_frame
    caption_frame.word_wrap = True
    p = caption_frame.paragraphs[0]
    p.text = 'Figure 7: Donut chart of most common file formats by total file count.'
    p.font.size = Pt(18)
    p.font.italic = True
    p.font.color.rgb = RGBColor(100, 100, 100)
    
    #####################################################
    # ===== SLIDE 10: File Format by Unique Dataset =====
    #####################################################

    slide = prs.slides.add_slide(prs.slide_layouts[6])

    # Add title
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(img_width), Inches(0.6))
    title_frame = title_box.text_frame
    p = title_frame.paragraphs[0]
    p.text = 'Most Common File Formats (by Unique Dataset)'
    p.font.size = Pt(32)
    p.font.bold = True
    p.font.color.rgb = RGBColor(*rgb_color)

    # Add figure with alt text
    fig_storage = figures['formats_by_dataset']
    img_stream = io.BytesIO()
    # fig_storage.savefig(img_stream, format='png',dpi = 100, bbox_inches='tight')
    fig_storage.savefig(img_stream, format='png',dpi = 100)
    img_stream.seek(0)
    picture_top = 1.1
    picture = slide.shapes.add_picture(img_stream, Inches(0.5), Inches(picture_top), width=Inches(img_width))
    picture.name = f'{institution_name} File Formats by Unique Dataset'
    picture._element._nvXxPr.cNvPr.attrib['descr'] = f'Donut graph comparing the 15 most common file formats by occurrence in unique datasets for {institution_name}.'

    # Add caption
    caption_y = get_figure_bottom_position(picture, picture_top) + 0.1
    caption_box = slide.shapes.add_textbox(Inches(0.5), Inches(caption_y), Inches(img_width), Inches(0.8))
    caption_frame = caption_box.text_frame
    caption_frame.word_wrap = True
    p = caption_frame.paragraphs[0]
    p.text = 'Figure 8: Donut chart of most common file formats by occurrence in unique datasets.'
    p.font.size = Pt(18)
    p.font.italic = True
    p.font.color.rgb = RGBColor(100, 100, 100)

    ###########################################
    # ===== SLIDE 11: Downloads and Views =====
    ###########################################

    slide = prs.slides.add_slide(prs.slide_layouts[6])

    # Add title
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(img_width), Inches(0.6))
    title_frame = title_box.text_frame
    p = title_frame.paragraphs[0]
    p.text = 'Dataset Downloads & Views'
    p.font.size = Pt(32)
    p.font.bold = True
    p.font.color.rgb = RGBColor(*rgb_color)

    # LEFT FIGURE: Downloads
    fig_downloads = figures['downloads']
    img_stream_left = io.BytesIO()
    fig_downloads.savefig(img_stream_left, format='png',dpi = 100, bbox_inches='tight')
    img_stream_left.seek(0)
    picture_top = 1.1
    picture_left = slide.shapes.add_picture(img_stream_left, Inches(0.5), Inches(picture_top), width=Inches(img_height))
    picture_left.name = f'{institution_name} Downloads'
    picture._element._nvXxPr.cNvPr.attrib['descr'] = f'Horizontal bar graph comparing dataset downloads by bin for {institution_name}.'

    # RIGHT FIGURE: Views
    fig_views = figures['views']
    img_stream_right = io.BytesIO()
    fig_views.savefig(img_stream_right, format='png',dpi = 100, bbox_inches='tight')
    img_stream_right.seek(0)
    picture_right = slide.shapes.add_picture(img_stream_right, Inches(7.5), Inches(picture_top), width=Inches(img_height))
    picture_right.name = f'{institution_name} Views'
    picture._element._nvXxPr.cNvPr.attrib['descr'] = f'Horizontal bar graph comparing dataset views by bin for {institution_name}.'

    # Add caption
    caption_y = get_figure_bottom_position(picture_left, picture_top) + 0.1
    caption_box = slide.shapes.add_textbox(Inches(0.5), Inches(caption_y), Inches(img_width), Inches(0.8))
    caption_frame = caption_box.text_frame
    caption_frame.word_wrap = True
    p = caption_frame.paragraphs[0]
    p.text = 'Figure 9: Distribution of dataset downloads and views.'
    p.font.size = Pt(18)
    p.font.italic = True
    p.font.color.rgb = RGBColor(100, 100, 100)

    ###############################################
    # ===== SLIDE 12: Top Datasets by Citations =====
    ###############################################

    slide = prs.slides.add_slide(prs.slide_layouts[6])

    # Add title
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(9), Inches(0.6))
    title_frame = title_box.text_frame
    p = title_frame.paragraphs[0]
    p.text = 'Top Datasets by Citations'
    p.font.size = Pt(32)
    p.font.bold = True
    p.font.color.rgb = RGBColor(*rgb_color)

    # Add datasets table
    table_top = 1.7
    table_shape = add_dataset_citation_table(
        slide=slide,
        datasets_df=dataset_citation_table,
        top_position=table_top,
        left_position=1,
        width=13,
        height=4.5
    )

    # Get actual bottom position from the shape's dimensions
    caption_y = 1.2
    caption_box = slide.shapes.add_textbox(Inches(1), Inches(caption_y), Inches(13), Inches(0.8))    
    caption_frame = caption_box.text_frame
    caption_frame.word_wrap = True
    p = caption_frame.paragraphs[0]
    p.text = 'Table 1: Top 10 most cited datasets (restricted to first 10 if there is a tie; only for datasets with at least one citation).'
    p.font.size = Pt(18)
    p.font.italic = True
    p.font.color.rgb = RGBColor(100, 100, 100)

    # Save presentation
    prs.save(output_path)
    print(f'✅ Presentation saved: {output_path}')

## Slide-deck creation

This codeblock contains the for-loop for recursively generating slide-decks for each institution.

In [6]:
#| echo: false
#| warning: false

# Omit 'Other affiliation'
dataverses_actual = dataverses_actual[dataverses_actual['institution_standardized'] != 'Other affiliation']

for institution in dataverses_actual['institution_standardized'].unique():
    print(f"\n🏫 Processing: {institution}")

    # Filter raw dataframes by institution
    dv_inst = dataverses_actual[dataverses_actual['institution_standardized'] == institution].copy()
    ds_inst = datasets[datasets['institution_standardized'] == institution].copy()
    usr_inst = users_real[users_real['Affiliation_Standardized'] == institution].copy()
    files_inst = files[files['institution'] == institution].copy()
    
    # ===== PROCESS DATAVERSES =====
    first_appearance_dv = dv_inst.groupby('contactIdentifier')['year_month_created'].min().reset_index()
    monthly_counts_dataverses = first_appearance_dv.groupby('year_month_created').size().reset_index(name='new_collections')
    monthly_counts_dataverses['total_collections'] = monthly_counts_dataverses['new_collections'].cumsum()
    
    # ===== PROCESS DATASETS =====
    first_appearance_ds = ds_inst.groupby('dataset_depositor')['year_month_created'].min().reset_index()
    monthly_counts_datasets = first_appearance_ds.groupby('year_month_created').size().reset_index(name='new_datasets')
    monthly_counts_datasets['total_datasets'] = monthly_counts_datasets['new_datasets'].cumsum()
    
    # Dataset publish counts
    monthly_counts = ds_inst.groupby('year_month_published').size()
    cumulative_counts = monthly_counts.cumsum()
    # cumulative_counts.index = cumulative_counts.index.to_timestamp()
    dataset_publish_counts = cumulative_counts.reset_index()
    dataset_publish_counts.columns = ['year_month', 'total']
    dataset_publish_counts['category'] = 'Published'
    
    # Dataset create counts
    monthly_counts = ds_inst.groupby('year_month_created').size()
    cumulative_counts = monthly_counts.cumsum()
    # cumulative_counts.index = cumulative_counts.index.to_timestamp()
    dataset_create_counts = cumulative_counts.reset_index()
    dataset_create_counts.columns = ['year_month', 'total']
    dataset_create_counts['category'] = 'Created'
    
    dataset_concat = pd.concat([dataset_publish_counts, dataset_create_counts], ignore_index=True)
    
    # ===== PROCESS USERS =====
    usr_inst['Created'] = pd.to_datetime(usr_inst['Created'])
    usr_inst['year_month_created'] = usr_inst['Created'].dt.to_period('M')
    monthly_counts = usr_inst.groupby('year_month_created').size()
    cumulative_counts = monthly_counts.cumsum()
    cumulative_counts.index = cumulative_counts.index.to_timestamp()
    users_summary = cumulative_counts.reset_index()
    users_summary.columns = ['year_month_created', 'total']
    users_summary['category'] = 'Users'
    
    # ===== COMBINE SUMMARIES FOR FIRST GRAPH =====
    datasets_summary = monthly_counts_datasets.copy()
    datasets_summary = datasets_summary[['year_month_created', 'total_datasets']]
    datasets_summary.columns = ['year_month_created', 'total']
    datasets_summary['category'] = 'Dataset creators'
    # datasets_summary['year_month_created'] = datasets_summary['year_month_created'].dt.to_timestamp()
    
    dataverses_summary = monthly_counts_dataverses.copy()
    dataverses_summary = dataverses_summary[['year_month_created', 'total_collections']]
    dataverses_summary.columns = ['year_month_created', 'total']
    dataverses_summary['category'] = 'Collection creators'
    # dataverses_summary['year_month_created'] = dataverses_summary['year_month_created'].dt.to_timestamp()
    
    combined_summary = pd.concat([users_summary, datasets_summary, dataverses_summary], ignore_index=True)
    combined_summary['year_month_created'] = pd.to_datetime(combined_summary['year_month_created'])
    
    # ===== PROCESS STORAGE DATA =====
    # Unpublished datasets
    datasets_copy = datasets[datasets['institution_standardized'] == institution].copy()
    datasets_copy = datasets_copy[datasets_copy['publicationDate'].isna()]
    datasets_copy['size_tb'] = datasets_copy['contentSize_MB'] / (1024 * 1024)
    datasets_copy['createTime'] = pd.to_datetime(datasets_copy['createTime'])
    datasets_copy['YearMonth'] = datasets_copy['createTime'].dt.to_period('M')
    monthly_counts_datasets = datasets_copy.groupby('YearMonth')['size_tb'].sum()
    cumulative_filesize_datasets = monthly_counts_datasets.cumsum()
    cumulative_filesize_datasets.index = cumulative_filesize_datasets.index.to_timestamp()
    
    # Published datasets from file-level metadata
    files_copy = files[files['institution'] == institution].copy()
    files_copy['size_gb'] = files_copy['file_size'] / (1024**3)
    files_copy['size_tb'] = files_copy['size_gb'] / 1024
    files_copy['file_creation_date'] = pd.to_datetime(files_copy['file_creation_date'])
    files_copy['YearMonth'] = files_copy['file_creation_date'].dt.to_period('M')
    monthly_counts = files_copy.groupby('YearMonth')['size_tb'].sum()
    cumulative_filesize = monthly_counts.cumsum()
    cumulative_filesize.index = cumulative_filesize.index.to_timestamp()
    
    # Align the indices
    all_dates = cumulative_filesize.index.union(cumulative_filesize_datasets.index)
    files_aligned = cumulative_filesize.reindex(all_dates, method='ffill').fillna(0)
    datasets_aligned = cumulative_filesize_datasets.reindex(all_dates, method='ffill').fillna(0)
    
    # Create storage summary dataframe for plotting
    storage_summary = pd.DataFrame({
        'year_month': files_aligned.index,
        'published': files_aligned.values,
        'unpublished': datasets_aligned.values
    })
    
    # ===== CREATE FIGURES =====
    ds_inst_fresh = datasets[datasets['institution_standardized'] == institution].copy()
    total_datasets = len(ds_inst_fresh)
    published_datasets = len(ds_inst_fresh[ds_inst_fresh['publicationDate'].notna()])
    unpublished_datasets = len(ds_inst_fresh[ds_inst_fresh['publicationDate'].isna()])
    total_dataverses = len(dv_inst)
    total_users = len(usr_inst)
    published_files=len(files_inst)
    published_storage = files_copy['size_tb'].sum().round(2)
    ds_inst_fresh['size_tb'] = ds_inst_fresh['contentSize_MB'] / (1024 * 1024)
    all_storage=ds_inst_fresh['size_tb'].sum().round(2)
    
    color_hex = school_colors[institution]
    fig_users = create_users_and_creators_trend(combined_summary, institution)
    fig_storage = create_storage_allocation_trend(storage_summary, institution)
    fig_file_formats = create_file_formats_pie(files_inst, institution, published_datasets)
    fig_file_formats_count = create_file_formats_by_count_pie(files_inst, institution)
    fig_collections = create_collections_trend(dv_inst, institution)
    fig_datasets = create_datasets_trend(ds_inst, institution)
    fig_datasets_subject = create_datasets_by_subject_bar(ds_inst, institution)
    fig_dataset_size = create_dataset_size_distribution(ds_inst, institution, bin_labels)
    fig_downloads = create_dataset_downloads_bin(ds_inst, institution)
    fig_views = create_dataset_views_bin(ds_inst, institution)

    # Add to figures dict
    figures = {
        'users': fig_users,
        'storage': fig_storage,
        'formats_by_dataset': fig_file_formats,
        'formats_by_count': fig_file_formats_count,
        'collections': fig_collections,
        'datasets': fig_datasets,
        'datasets_subject': fig_datasets_subject,
        'dataset_size': fig_dataset_size,
        'downloads': fig_downloads,
        'views': fig_views,
    }

    # Store most cited table separately
    dataset_citation_table = ds_inst.sort_values('citations_dc', ascending=False)[['dataset_title', 'citations_dc', 'persistentUrl']]   
    
    # ===== CREATE PRESENTATION =====
    output_path = os.path.join(reports_dir, f"{institution.replace(' ', '_')}_annual-report_{current_year}.pptx")
    create_presentation(
        institution_name=institution,
        display_names=display_names,
        figures=figures,
        dataset_citation_table=dataset_citation_table,
        output_path=output_path,
        institution_color=color_hex,
        total_datasets=total_datasets,
        published_datasets=published_datasets,
        unpublished_datasets=unpublished_datasets,
        total_dataverses=total_dataverses,
        total_users=total_users,
        published_files=published_files,
        all_storage=all_storage,
        published_storage=published_storage
    )

    for fig in figures.values():
        plt.close(fig)

print("\n✅ All presentations generated!")


🏫 Processing: Baylor U


C:\Users\bmg3525\AppData\Local\Temp\ipykernel_21044\3102165354.py:45: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  usr_inst['Created'] = pd.to_datetime(usr_inst['Created'])
C:\Users\bmg3525\AppData\Local\Temp\ipykernel_21044\3102165354.py:76: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  datasets_copy['YearMonth'] = datasets_copy['createTime'].dt.to_period('M')


✅ Presentation saved: institutional-reports\Baylor_U_annual-report_2026.pptx

🏫 Processing: Lamar U


C:\Users\bmg3525\AppData\Local\Temp\ipykernel_21044\3102165354.py:45: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  usr_inst['Created'] = pd.to_datetime(usr_inst['Created'])
C:\Users\bmg3525\AppData\Local\Temp\ipykernel_21044\3513594893.py:270: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=legend_font, loc='upper left')


✅ Presentation saved: institutional-reports\Lamar_U_annual-report_2026.pptx

🏫 Processing: SMU


...

✅ Presentation saved: institutional-reports\UT_Austin_annual-report_2026.pptx

🏫 Processing: UTHSCSA


C:\Users\bmg3525\AppData\Local\Temp\ipykernel_21044\3102165354.py:45: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  usr_inst['Created'] = pd.to_datetime(usr_inst['Created'])
C:\Users\bmg3525\AppData\Local\Temp\ipykernel_21044\3102165354.py:76: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  datasets_copy['YearMonth'] = datasets_copy['createTime'].dt.to_period('M')


✅ Presentation saved: institutional-reports\UTHSCSA_annual-report_2026.pptx

🏫 Processing: UT Southwestern


C:\Users\bmg3525\AppData\Local\Temp\ipykernel_21044\3102165354.py:45: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  usr_inst['Created'] = pd.to_datetime(usr_inst['Created'])
C:\Users\bmg3525\AppData\Local\Temp\ipykernel_21044\3102165354.py:76: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  datasets_copy['YearMonth'] = datasets_copy['createTime'].dt.to_period('M')


✅ Presentation saved: institutional-reports\UT_Southwestern_annual-report_2026.pptx

✅ All presentations generated!
